In [1]:
import os
import pandas as pd
import numpy as np
import glob

from datetime import datetime, timedelta
from astropy.time import Time
import gc

from scipy.ndimage import shift

# functions

In [2]:
def remove_duplicate_glitches(df_clustered, glitchgrams_list, threshold_snr, pixel_time_res=0.04878):

    if len(df_clustered) <= 1:
        return df_clustered

    df_work = df_clustered.copy()
    df_work['orig_idx'] = np.arange(len(df_work))
    df_sorted = df_work.sort_values(by='time').reset_index(drop=True)
    
    to_drop = set()

    last_valid_pos = 0

    for i in range(1, len(df_sorted)):
        prev_row = df_sorted.iloc[last_valid_pos]
        curr_row = df_sorted.iloc[i]

        time_prev = float(prev_row['time'])
        time_curr = float(curr_row['time'])
        
        dt = time_curr - time_prev

        if 0 < dt <= gap_time:
            idx_prev = int(prev_row['orig_idx'])
            idx_curr = int(curr_row['orig_idx'])

            img_prev = glitchgrams_list[idx_prev]
            img_curr = glitchgrams_list[idx_curr]

            max_p = np.max(img_prev)
            max_c = np.max(img_curr)

            if max_p > 0 and max_c > 0:
                norm_prev = img_prev / max_p
                norm_curr = img_curr / max_c

                shift_pixels = int(round(dt / pixel_time_res))
                img_shifted = shift(norm_curr, shift=[0, -shift_pixels], mode='constant', cval=0)

                mask = (norm_prev > 0.1) | (img_shifted > 0.1)

                if np.sum(mask) >= 1:
                    diff_snr = np.mean(np.abs(norm_prev[mask] - img_shifted[mask]))
    
                    if diff_snr < threshold_snr:
                        snr_prev = float(np.asarray(prev_row['snr']).squeeze())
                        snr_curr = float(np.asarray(curr_row['snr']).squeeze())

                        if snr_curr > snr_prev:
                            to_drop.add(idx_prev)
                            last_valid_pos = i
                        else:
                            to_drop.add(idx_curr)
                        continue
        last_valid_pos = i
    
    df_cleaned = df_sorted[~df_sorted['orig_idx'].isin(to_drop)].drop(columns=['orig_idx']).reset_index(drop=True)
    return df_cleaned

In [3]:
def get_glitchgrams_for_day(unclustered, clustered, window_T=2.0, fmin=3, fmax=8192, tbins=41, fbins=30):

    all_glitchgrams = []

    unclustered = unclustered.sort_values("time")
    
    times_all = unclustered['time'].values
    freqs_all = unclustered['frequency'].values
    snrs_all  = unclustered['snr'].values

    time_edges = np.linspace(-window_T/2, window_T/2, tbins+1)
    freq_edges = np.logspace(np.log10(fmin), np.log10(fmax), fbins+1)
    
    for i, row in enumerate(clustered.itertuples(index=False)):

        t_center = row.time
    
        tmin = t_center - window_T/2
        tmax = t_center + window_T/2
    
        i0 = np.searchsorted(times_all, tmin, side="left")
        i1 = np.searchsorted(times_all, tmax, side="right")
    
        times = times_all[i0:i1] - t_center
        freqs = freqs_all[i0:i1]
        snrs  = snrs_all[i0:i1]

        t_idx = np.digitize(times, time_edges) - 1
        f_idx = np.digitize(freqs, freq_edges) - 1
            
        A = np.zeros((fbins, tbins), dtype=float)
    
        for ti, fi, si in zip(t_idx, f_idx, snrs):
            if 0 <= ti < tbins and 0 <= fi < fbins:
                if si > A[fi, ti]:
                    A[fi, ti] = si

        all_glitchgrams.append(A)
        
    return all_glitchgrams

In [4]:
def clusteringGlitches(gltab, min_triggers, temp_dist):
    
    if len(gltab) == 0:
        return pd.DataFrame()

    table = gltab.sort_values(by='time').reset_index(drop=True).copy()

    times = table['time'].values
    deltas = np.diff(times)

    new_cluster_mask = np.concatenate([[False], deltas > temp_dist])
    table['cluster'] = np.cumsum(new_cluster_mask)

    counts = table['cluster'].value_counts().sort_index()
    
    valid_clusters = counts[counts >= min_triggers].index
    table_filtered = table[table['cluster'].isin(valid_clusters)].copy()

    if table_filtered.empty:
        return pd.DataFrame()

    unique_clusters = table_filtered['cluster'].unique()
    cluster_mapping = {old_id: new_id for new_id, old_id in enumerate(unique_clusters)}
    table_filtered['cluster'] = table_filtered['cluster'].map(cluster_mapping)

    df_clustered = table_filtered.copy()
    
    if df_clustered.empty:
        return pd.DataFrame()

    summary_list = []

    for cl, group in df_clustered.groupby('cluster'):
        # Encontra a linha/trigger com o maior SNR do grupo
        max_row = group.loc[group['snr'].idxmax()]

        summary_list.append({
            'cluster': cl,
            'num_triggers': len(group),
            'time': max_row['time'],             # GPS Time oficial (dt=0 no pico SNR)
            'frequency': max_row['frequency'],   # Frequência central no pico
            'snr': max_row['snr'],               # SNR Máximo
            'tstart': group['time'].min(),
            'tend': group['time'].max(),
            'min_freq': group['frequency'].min(),
            'max_freq': group['frequency'].max()
        })

    return pd.DataFrame(summary_list)

In [17]:
def filter_intervals(triggers, gps_start, gps_end, intervals):
    
    intervals_crop = intervals[(intervals['int2'] >= gps_start) & (intervals['int1'] <= gps_end)].copy().sort_values('int1')
    intervals_crop['int1'] = intervals_crop['int1'].astype('float64')

    triggers_crop = triggers[(triggers['time'] >= gps_start) & (triggers['time'] <= gps_end)].copy().sort_values('time')

    if triggers_crop.empty or intervals_crop.empty:
        return pd.DataFrame(columns=triggers.columns)

    merged = pd.merge_asof(triggers_crop, intervals_crop, left_on='time', right_on='int1', direction='backward')
    valid = merged[(merged['time'] >= merged['int1']) & (merged['time'] <= merged['int2'])]
    # valid = merged[merged['time'] <= merged['int2']].copy()

    triggers_final = valid.drop(columns=['int1', 'int2'], errors='ignore')

    return triggers_final

In [6]:
def routine(current_day, end_day, unclustered, intervals, min_triggers, deltat, 
            gap_time, threshold_snr, output_dir=None, global_cluster_ID=0):
    
    os.makedirs(output_dir, exist_ok=True)
    trigger_count = []
    glitch_count = []

    all_glitches = []
    
    while current_day <= end_day:
        
        date_str = current_day.strftime('%Y-%m-%d')
        out_file = os.path.join(output_dir, f"glitches_{date_str}.parquet")

        if os.path.exists(out_file):
            print(f"Day {date_str} already processed. Skipping...")

            try:
                df_exist = pd.read_parquet(out_file)
                if not df_exist.empty:
                    global_cluster_ID = df_exist['cluster'].max() + 1
            except Exception:
                pass
            
            current_day += timedelta(days=1)
            continue

        day_start = datetime(current_day.year, current_day.month, current_day.day, 0, 0, 0)
        day_end   = datetime(current_day.year, current_day.month, current_day.day, 23, 59, 59)

        print(f"Processing day: {current_day.date()}")

        gps_start = Time(day_start, scale="utc").gps
        gps_end   = Time(day_end, scale="utc").gps

        # CONSIDER THE INTERVALS OF THE INTERFEROMETER ------------------------------
        gltab = unclustered[(unclustered['time'] >= gps_start) & (unclustered['time'] <= gps_end)].copy()
        
        gltab = filter_intervals(gltab, gps_start, gps_end, intervals)
        gltab = gltab.sort_values(by="time").reset_index(drop=True)

        # CLUSTERING OF THE CURRENT DAY
        if not gltab.empty:
            # CLUSTERING OF THE CURRENT DAY ------------------------------
            clustered = clusteringGlitches(gltab, min_triggers, deltat)

        if not clustered.empty:
            # CREATE GLITCHGRAMS FOR CURRENT DAY ------------------------------
            # glitchgrams_dict, rel_times = get_glitchgrams_for_day(gltab, clustered)
            glitchgrams_dict = get_glitchgrams_for_day(gltab, clustered)
    
            # REMOVE SUBSEQUENT DUPLICATE GLITCHES ------------------------------
            clustered = remove_duplicate_glitches(clustered, glitchgrams_dict, threshold_snr)
    
            if not clustered.empty:
                # CREATE GLITCHGRAMS FOR CURRENT DAY ------------------------------
                clustered['cluster'] += global_cluster_ID
                global_cluster_ID = clustered['cluster'].max() + 1
                
                abs_out_file = os.path.abspath(out_file)
        
                try:
                    clustered.to_parquet(abs_out_file, index=False)
                    print(f"Successfully saved: {out_file}\n")
                except Exception as e:
                    csv_file = abs_out_file.replace('.parquet', '.csv')
                    clustered.to_csv(csv_file, index=False)
                    print(f"Parquet failed ({e}). SAVED IN CSV: {csv_file}\n")
    
                all_glitches.append(clustered.copy())
                del clustered
            else:
                print(f"No valid clusters found for day {date_str}.\n")
                gltab = gltab.sort_values(by="time").reset_index(drop=True)
        
        del gltab
        gc.collect()
            
        current_day += timedelta(days=1)
        
    print(f"Processing complete. Next available global_cluster_ID: {global_cluster_ID}")
    
    if all_glitches:
        return pd.concat(all_glitches, ignore_index=True)
    else:
        return pd.DataFrame()

# routine

## Parameters and Variables

In [57]:
window_T = 2.0       # window around the glitch
deltat = 0.2         # time btw triggers from the same cluster 

cutSNR = True
SNRvalue_cut = 7.5
min_triggers = 40

gap_time = 2.0        # time btw possible duplicated glitches
threshold_snr = 1.0

In [58]:
run = 'O3a'
clust = 'teste5_duplicate_glitches'
arqType = 'SNRclustered'

suffix = "_C" + str(min_triggers) + "_dT" + str(deltat) + "_SNR" + str(SNRvalue_cut) + "_gt" + str(gap_time) + "_th" + str(threshold_snr)

In [59]:
base = "/home/esanches/Projetos/tSNE/Virgo"
repository = "/home/esanches/Projetos/tSNE/Virgo/zrepository"
daily_clustering = "/home/esanches/Projetos/tSNE/Virgo/zrepository/" + clust + "/" + run + "/" + "daily_clustering" + suffix

daily_clustering

'/home/esanches/Projetos/tSNE/Virgo/zrepository/teste5_duplicate_glitches/O3a/daily_clustering_C40_dT0.001_SNR7.5_gt2.0_th1.0'

## Read the triggers

In [15]:
trigcols = ['time', 'frequency', 'snr']

dtypes = {
    'time': 'float64',
    'frequency': 'float32',
    'snr': 'float32'
}

path = base + "/" + "triggers_total_" + run + ".csv"

chunk_size = 10_000_000
count = 0
chunks = []

for chunk in pd.read_csv(path, usecols=trigcols, dtype=dtypes, chunksize=chunk_size):
    print(f'Reading chunk: {count}')
    chunks.append(chunk)
    count += 1

    del chunk
    gc.collect()

triggers = pd.concat(chunks, ignore_index=True)
triggers = triggers.sort_values('time').reset_index(drop=True)

unclustered = triggers.copy()

unclustered.tail()

Reading chunk: 0
Reading chunk: 1
Reading chunk: 2
Reading chunk: 3
Reading chunk: 4
Reading chunk: 5
Reading chunk: 6
Reading chunk: 7
Reading chunk: 8
Reading chunk: 9
Reading chunk: 10
Reading chunk: 11
Reading chunk: 12
Reading chunk: 13
Reading chunk: 14
Reading chunk: 15


,time,frequency,snr
156426272,1.253977e+09,22.160860,5.007397
156426273,1.253977e+09,49.533298,5.252314
156426274,1.253977e+09,74.707069,5.137199
156426275,1.253977e+09,60.831635,5.173639
156426276,1.253977e+09,49.533298,5.312661


## The routine

In [60]:
start_day_O3a = datetime(2019, 4, 1, 0, 0, 0)
end_day_O3a   = datetime(2019, 10, 1, 15, 0, 0)

start = start_day_O3a
end = end_day_O3a

intervals = pd.read_csv(repository + '/gwosc' + run + '.txt', sep=' ', header=None, names=['int1', 'int2', 'duration'])

df_glitches_final = routine(start, end, unclustered, intervals, min_triggers, deltat, gap_time, threshold_snr, output_dir=daily_clustering)

Processing day: 2019-04-01
Successfully saved: /home/esanches/Projetos/tSNE/Virgo/zrepository/teste5_duplicate_glitches/O3a/daily_clustering_C40_dT0.001_SNR7.5_gt2.0_th1.0/glitches_2019-04-01.parquet

Processing day: 2019-04-02
Successfully saved: /home/esanches/Projetos/tSNE/Virgo/zrepository/teste5_duplicate_glitches/O3a/daily_clustering_C40_dT0.001_SNR7.5_gt2.0_th1.0/glitches_2019-04-02.parquet

Processing day: 2019-04-03
Successfully saved: /home/esanches/Projetos/tSNE/Virgo/zrepository/teste5_duplicate_glitches/O3a/daily_clustering_C40_dT0.001_SNR7.5_gt2.0_th1.0/glitches_2019-04-03.parquet

Processing day: 2019-04-04
Successfully saved: /home/esanches/Projetos/tSNE/Virgo/zrepository/teste5_duplicate_glitches/O3a/daily_clustering_C40_dT0.001_SNR7.5_gt2.0_th1.0/glitches_2019-04-04.parquet

Processing day: 2019-04-05
Successfully saved: /home/esanches/Projetos/tSNE/Virgo/zrepository/teste5_duplicate_glitches/O3a/daily_clustering_C40_dT0.001_SNR7.5_gt2.0_th1.0/glitches_2019-04-05.parq

## Combine and save the .parquet files

These will put together the data obtained from Routine in only one .csv arquive

In [61]:
arq_name = "SNRclustered" + suffix if cutSNR else "clustered" + suffix - "_"
arquivo_csv_saida = repository + "/" + clust + "/" + run + "/" + arq_name + ".csv"

arquivos_parquet = sorted(glob.glob(os.path.join(daily_clustering, "glitches_*.parquet")))

df_list = [pd.read_parquet(f) for f in arquivos_parquet]
df_glitches_final = pd.concat(df_list, ignore_index=True)

df_glitches_final = df_glitches_final.drop(columns=['cluster']).sort_values('time').reset_index(drop=True)

if cutSNR:
    df_glitches_final = df_glitches_final[df_glitches_final['snr'] >= SNRvalue_cut].sort_values('time').reset_index(drop=True)

df_glitches_final.to_csv(arquivo_csv_saida, index=False)

df_glitches_final

,num_triggers,time,frequency,snr,tstart,tend,min_freq,max_freq
0,109,1238166408.178711,112.674637,11.276587,1238166408.171875,1238166408.183593,49.533298,386.562347
1,128,1238168111.496093,343.490845,10.960562,1238168111.474609,1238168111.499023,256.303833,389.409332
2,109,1238168121.833984,346.784241,12.343865,1238168121.829101,1238168121.842773,256.303833,583.020752
3,183,1238168134.408203,346.784241,13.739071,1238168134.398437,1238168134.417968,159.007568,583.020752
4,267,1238168137.255859,346.784241,17.667526,1238168137.236328,1238168137.269531,256.303833,583.020752
...,...,...,...,...,...,...,...,...
14098,6860,1253965386.369140,74.707069,558.574341,1253965386.259765,1253965386.476562,16.116238,1942.651001
14099,132,1253966582.675781,26.742411,21.374872,1253966582.673828,1253966582.683593,17.731117,474.735565
14100,22410,1253968667.313598,1079.892944,149.980255,1253968667.222656,1253968667.499023,17.731117,1942.651001
14101,26680,1253972787.922607,1149.036743,455.738190,1253972787.738281,1253972788.166015,16.457972,1942.651001


In [62]:
len(df_glitches_final)

14103

In [63]:
df_glitches_final['num_triggers'].max()

np.int64(44613)

In [64]:
maximum = df_glitches_final[df_glitches_final['num_triggers'] == df_glitches_final['num_triggers'].max()]

In [65]:
pd.set_option('display.float_format', lambda x: '%.6f' % x)

t_inicio = maximum['tstart'].values[0]
t_fim = maximum['tend'].values[0]
duracao = t_fim - t_inicio

print(f'Início (GPS): {t_inicio}')
print(f'Fim (GPS):    {t_fim}')
print(f'Duração:      {duracao:.4f} segundos')

Início (GPS): 1242311026.191406
Fim (GPS):    1242311027.749023
Duração:      1.5576 segundos
